# 02 — Publish edges to Purview

Reads the latest edges from `lineage_edges` and upserts them to Purview Atlas
v2 as typed entities + `Process` nodes. Idempotent on `qualifiedName`.

Runs as the second step of the `lineage_harvest_pipeline` Fabric Data Pipeline.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path("/lakehouse/default/Files/repo/fabric-lineage-graph")
if REPO_ROOT.exists():
    sys.path.insert(0, str(REPO_ROOT))

from pyspark.sql import SparkSession
from common.schema import LineageEdge
from graph.push_to_purview import push_edges

spark = SparkSession.builder.getOrCreate()
rows = [r.asDict(recursive=True) for r in spark.table("lineage_edges").collect()]
print(f"Loaded {len(rows)} edges from lineage_edges")

In [ ]:
edges = [LineageEdge(
    source_qname=r["source_qname"], source_type=r["source_type"],
    target_qname=r["target_qname"], target_type=r["target_type"],
    process_name=r["process_name"], process_type=r["process_type"],
    artifact_ref=r["artifact_ref"],
    harvested_at=str(r["harvested_at"]),
) for r in rows]

result = push_edges(edges)
print(f"Purview upsert: {result}")